In [1]:
!pip install psycopg2-binary sqlalchemy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 65.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

In [5]:
DB_USER = 'postgres'
DB_PASSWORD = 'password' # Replace with your PostgreSQL password
DB_HOST = 'localhost'
DB_PORT = '5432'

# Connect to the default 'postgres' database to create new databases
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/postgres"
engine = create_engine(DATABASE_URL)

In [6]:
# Connect to the engine and create the database
# Using 'text' for raw SQL statements
try:
    with engine.connect() as connection:
        connection.execute(text("CREATE DATABASE al_exam_dw;"))
        connection.commit() # Commit the transaction
    print("Database 'al_exam_dw' created successfully.")
except Exception as e:
    if "already exists" in str(e):
        print("Database 'al_exam_dw' already exists.")
    else:
        print(f"An error occurred: {e}")

An error occurred: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [3]:
CREATE DATABASE al_exam_dw;

SyntaxError: invalid syntax (1554626206.py, line 1)

### Switching to SQLite Database

Since connecting to an external PostgreSQL server failed and you want to work within this environment, we'll use SQLite. SQLite is a serverless, self-contained, high-reliability, full-featured, SQL database engine. It saves the database to a file (in this case, `al_exam_dw.db`) directly on the Colab instance, making it perfect for local development and testing.

In [7]:
# Connect to the new SQLite database file 'al_exam_dw.db'
# This will create the database file if it doesn't exist.

DATABASE_URL = f"sqlite:///al_exam_dw.db"
engine = create_engine(DATABASE_URL)

print("SQLite database 'al_exam_dw.db' connected (and created if it didn't exist).")

SQLite database 'al_exam_dw.db' connected (and created if it didn't exist).


### Creating Tables in the SQLite Database

Now that the SQLite database is ready, let's create some example tables. I'll create `customers`, `products`, and `orders` tables. You can modify these schemas as needed for your specific use case.

In [8]:
try:
    with engine.connect() as connection:
        # Create 'customers' table
        connection.execute(text("""
            CREATE TABLE IF NOT EXISTS customers (
                customer_id INTEGER PRIMARY KEY,
                first_name VARCHAR(50) NOT NULL,
                last_name VARCHAR(50) NOT NULL,
                email VARCHAR(100) UNIQUE
            );
        """))
        # Create 'products' table
        connection.execute(text("""
            CREATE TABLE IF NOT EXISTS products (
                product_id INTEGER PRIMARY KEY,
                product_name VARCHAR(100) NOT NULL,
                price DECIMAL(10, 2) NOT NULL
            );
        """))
        # Create 'orders' table
        connection.execute(text("""
            CREATE TABLE IF NOT EXISTS orders (
                order_id INTEGER PRIMARY KEY,
                customer_id INTEGER NOT NULL,
                order_date DATE NOT NULL,
                total_amount DECIMAL(10, 2) NOT NULL,
                FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
            );
        """))
        connection.commit()
    print("Tables 'customers', 'products', and 'orders' created successfully (or already exist).")
except Exception as e:
    print(f"An error occurred during table creation: {e}")

Tables 'customers', 'products', and 'orders' created successfully (or already exist).


### Creating Custom Dimension and Fact Tables

As requested, I will now create your specified dimension tables (`dim_student`, `dim_subject`, `dim_stream`, `dim_location`, `dim_date`, `dim_grade`) and the fact table (`fact_exam_result`). This will replace the previous `customers`, `products`, and `orders` tables.

In [11]:
create_dimensions = """
CREATE TABLE IF NOT EXISTS dim_student (
    student_key INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id VARCHAR(30) UNIQUE,
    gender VARCHAR(20),
    date_of_birth DATE
);

CREATE TABLE IF NOT EXISTS dim_subject (
    subject_key INTEGER PRIMARY KEY AUTOINCREMENT,
    subject_code VARCHAR(20),
    subject_name VARCHAR(100),
    subject_category VARCHAR(50)
);

CREATE TABLE IF NOT EXISTS dim_stream (
    stream_key INTEGER PRIMARY KEY AUTOINCREMENT,
    stream_code VARCHAR(20),
    stream_name VARCHAR(100),
    stream_category VARCHAR(100)
);

CREATE TABLE IF NOT EXISTS dim_location (
    location_key INTEGER PRIMARY KEY AUTOINCREMENT,
    district VARCHAR(100),
    province VARCHAR(100),
    examination_region VARCHAR(100)
);

CREATE TABLE IF NOT EXISTS dim_date (
    date_key INT PRIMARY KEY,
    full_date DATE,
    day INT,
    month INT,
    month_name VARCHAR(20),
    quarter INT,
    year INT
);

CREATE TABLE IF NOT EXISTS dim_grade (
    grade_key INTEGER PRIMARY KEY AUTOINCREMENT,
    grade_code VARCHAR(10),
    grade_description VARCHAR(50),
    grade_points DECIMAL(5,2),
    pass_flag BOOLEAN
);
"""


# Adjusting for SQLite: SERIAL becomes INTEGER PRIMARY KEY AUTOINCREMENT
# Remove SERIAL keyword and replace with INTEGER PRIMARY KEY AUTOINCREMENT for SQLite compatibility
# Also, split into individual statements for SQLite as it doesn't support multiple statements in one execute call
create_dimensions_sqlite = create_dimensions.replace('SERIAL PRIMARY KEY', 'INTEGER PRIMARY KEY AUTOINCREMENT')

# Split SQL string into individual statements and execute them
with engine.begin() as conn:
    for statement in create_dimensions_sqlite.split(';')[:-1]: # Split by ';' and exclude the last empty string
        if statement.strip(): # Ensure statement is not empty
            conn.execute(text(statement))

print("Dimension tables created successfully.")

Dimension tables created successfully.


In [10]:
create_fact = """
CREATE TABLE IF NOT EXISTS fact_exam_result (
    result_key INTEGER PRIMARY KEY AUTOINCREMENT,

    student_key INT NOT NULL,
    subject_key INT NOT NULL,
    stream_key INT NOT NULL,
    location_key INT,
    date_key INT NOT NULL,
    grade_key INT,

    marks DECIMAL(5,2),
    z_score DECIMAL(6,3),

    pass_flag BOOLEAN,
    attempt_number INT,

    candidate_count INT,

    CONSTRAINT fk_student
        FOREIGN KEY (student_key)
        REFERENCES dim_student(student_key),

    CONSTRAINT fk_subject
        FOREIGN KEY (subject_key)
        REFERENCES dim_subject(subject_key),

    CONSTRAINT fk_stream
        FOREIGN KEY (stream_key)
        REFERENCES dim_stream(stream_key),

    CONSTRAINT fk_location
        FOREIGN KEY (location_key)
        REFERENCES dim_location(location_key),

    CONSTRAINT fk_date
        FOREIGN KEY (date_key)
        REFERENCES dim_date(date_key),

    CONSTRAINT fk_grade
        FOREIGN KEY (grade_key)
        REFERENCES dim_grade(grade_key)
);
"""

# Adjusting for SQLite: BIGSERIAL becomes INTEGER PRIMARY KEY AUTOINCREMENT
create_fact_sqlite = create_fact.replace('BIGSERIAL PRIMARY KEY', 'INTEGER PRIMARY KEY AUTOINCREMENT')


with engine.begin() as conn:
    conn.execute(text(create_fact_sqlite))

print("Fact table created successfully.")

Fact table created successfully.


In [12]:
from google.colab import files

uploaded = files.upload()


Saving 2020_al_data_kaggle_upload_new_old_syllabi.csv to 2020_al_data_kaggle_upload_new_old_syllabi.csv


In [17]:
import pandas as pd
import io

# Load the single uploaded raw data file
# The 'uploaded' variable from cell hKtnRS1Kgm4c contains the content of the uploaded file(s).
# Assuming '2020_al_data_kaggle_upload_new_old_syllabi.csv' was the file uploaded.
file_name = '2020_al_data_kaggle_upload_new_old_syllabi.csv'
if file_name in uploaded:
    file_content = uploaded[file_name]
    main_df = pd.read_csv(io.BytesIO(file_content))
    print(f"Successfully loaded '{file_name}' into 'main_df'. Shape: {main_df.shape}")

    # Explicitly set column names to ensure consistency and prevent KeyError
    # Based on the header from the uploaded CSV:
    # index,stream,Zscore,district_rank,island_rank,al_year,sub1,sub1_r,sub2,sub2_r,sub3,sub3_r,pass_fail,total_marks,grade,gender,day,month,year
    expected_columns = [
        'index', 'stream', 'Zscore', 'district_rank', 'island_rank', 'al_year',
        'sub1', 'sub1_r', 'sub2', 'sub2_r', 'sub3', 'sub3_r', 'pass_fail',
        'total_marks', 'grade', 'gender', 'day', 'month', 'year'
    ]
    if len(main_df.columns) == len(expected_columns):
        main_df.columns = expected_columns
        print("Main_df columns explicitly renamed for consistency.")
    else:
        print(f"Warning: Column count mismatch. Expected {len(expected_columns)}, got {len(main_df.columns)}. Column renaming skipped.")
        print("Actual columns found:", main_df.columns.tolist())

else:
    print(f"Error: '{file_name}' not found in uploaded files. Please ensure it's uploaded and the name is correct.")
    main_df = pd.DataFrame() # Initialize empty to prevent further errors

# The original code attempted to load separate CSVs for each dimension and fact table.
# Since only one raw data CSV was uploaded, these tables need to be derived
# from the 'main_df'. The following lines are placeholders and need to be
# replaced with actual data extraction and transformation logic.

# Dimension tables - placeholders for extraction from main_df
student_df = pd.DataFrame(columns=['student_key', 'student_id', 'gender', 'date_of_birth'])
subject_df = pd.DataFrame(columns=['subject_key', 'subject_code', 'subject_name', 'subject_category'])
stream_df = pd.DataFrame(columns=['stream_key', 'stream_code', 'stream_name', 'stream_category'])
location_df = pd.DataFrame(columns=['location_key', 'district', 'province', 'examination_region'])
date_df = pd.DataFrame(columns=['date_key', 'full_date', 'day', 'month', 'month_name', 'quarter', 'year'])
grade_df = pd.DataFrame(columns=['grade_key', 'grade_code', 'grade_description', 'grade_points', 'pass_flag'])

# Fact table - placeholder for extraction and merging with dimension keys
fact_df = pd.DataFrame(columns=[
    'result_key', 'student_key', 'subject_key', 'stream_key', 'location_key',
    'date_key', 'grade_key', 'marks', 'z_score', 'pass_flag',
    'attempt_number', 'candidate_count'
])

if not main_df.empty:
    print("Placeholders created for dimension and fact DataFrames. You now need to populate them from 'main_df'.")
    print("Example: student_df = main_df[['some_student_id_col', 'gender_col', 'dob_col']].drop_duplicates().reset_index(drop=True)")
    print("Please provide instructions on how to extract data from 'main_df' for each dimension and the fact table.")
else:
    print("Main DataFrame is empty due to upload error. Please re-upload your file.")


Successfully loaded '2020_al_data_kaggle_upload_new_old_syllabi.csv' into 'main_df'. Shape: (337553, 19)
Main_df columns explicitly renamed for consistency.
Placeholders created for dimension and fact DataFrames. You now need to populate them from 'main_df'.
Example: student_df = main_df[['some_student_id_col', 'gender_col', 'dob_col']].drop_duplicates().reset_index(drop=True)
Please provide instructions on how to extract data from 'main_df' for each dimension and the fact table.


In [19]:
# --- 1. Populate dim_student --- #
student_df_raw = main_df[['gender', 'day', 'month', 'year']].drop_duplicates().copy()
student_df_raw.columns = ['gender', 'dob_day', 'dob_month', 'dob_year']

# Handle potential missing/invalid date parts for DOB
student_df_raw['date_of_birth_str'] = student_df_raw['dob_year'].astype(str) + '-' + \
                                    student_df_raw['dob_month'].astype(str).str.zfill(2) + '-' + \
                                    student_df_raw['dob_day'].astype(str).str.zfill(2)

student_df_raw['date_of_birth'] = pd.to_datetime(student_df_raw['date_of_birth_str'], errors='coerce')

# Create a unique student_id based on gender and date_of_birth
# If date_of_birth is null, use a placeholder for ID generation
student_df_raw['student_id'] = student_df_raw.apply(
    lambda row: f"{row['gender']}_{row['date_of_birth'].strftime('%Y%m%d')}"
    if pd.notna(row['date_of_birth']) else f"{row['gender']}_UNKNOWN_{row['dob_day']}_{row['dob_month']}_{row['dob_year']}",
    axis=1
)

student_df = student_df_raw[['student_id', 'gender', 'date_of_birth']].copy()
student_df.dropna(subset=['student_id'], inplace=True)
student_df.reset_index(drop=True, inplace=True)
print(f"dim_student created. Rows: {len(student_df)}")


# --- 2. Populate dim_subject --- #
subjects = pd.concat([main_df['sub1'], main_df['sub2'], main_df['sub3']]).unique()
subjects = pd.DataFrame(subjects, columns=['subject_code'])
subjects.dropna(subset=['subject_code'], inplace=True)
subject_df = subjects.copy()
subject_df['subject_name'] = subject_df['subject_code'] # Assuming name is same as code for now
subject_df['subject_category'] = None # No info in main_df
subject_df.reset_index(drop=True, inplace=True)
print(f"dim_subject created. Rows: {len(subject_df)}")


# --- 3. Populate dim_stream --- #
stream_df = main_df[['stream']].drop_duplicates().copy()
stream_df.rename(columns={'stream': 'stream_code'}, inplace=True)
stream_df.dropna(subset=['stream_code'], inplace=True)
stream_df['stream_name'] = stream_df['stream_code'] # Assuming name is same as code
stream_df['stream_category'] = None # No info in main_df
stream_df.reset_index(drop=True, inplace=True)
print(f"dim_stream created. Rows: {len(stream_df)}")


# --- 4. Populate dim_location --- #
location_df = main_df[['district_rank', 'island_rank']].drop_duplicates().copy()
location_df.rename(columns={'district_rank': 'district'}, inplace=True)
location_df['province'] = None # No info in main_df
location_df['examination_region'] = location_df['island_rank'] # Using island_rank for region
location_df.drop(columns=['island_rank'], inplace=True)
location_df.dropna(subset=['district'], inplace=True)
location_df.reset_index(drop=True, inplace=True)
print(f"dim_location created. Rows: {len(location_df)}")


# --- 5. Populate dim_date --- #
# Assuming al_year is the examination year
date_df_raw = main_df[['al_year']].drop_duplicates().copy()
date_df_raw.rename(columns={'al_year': 'year'}, inplace=True)
date_df_raw.dropna(subset=['year'], inplace=True)

# Create full dates for the exam year (e.g., start of the year)
# For simplicity, let's assume the date refers to the exam year, not specific day/month from main_df for exam date
date_df_raw['full_date'] = pd.to_datetime(date_df_raw['year'].astype(str) + '-01-01', errors='coerce')
date_df = date_df_raw.copy()
date_df['date_key'] = date_df['full_date'].dt.strftime('%Y%m%d').astype(int)
date_df['day'] = date_df['full_date'].dt.day
date_df['month'] = date_df['full_date'].dt.month
date_df['month_name'] = date_df['full_date'].dt.month_name()
date_df['quarter'] = date_df['full_date'].dt.quarter
date_df.dropna(subset=['date_key'], inplace=True)
date_df = date_df[['date_key', 'full_date', 'day', 'month', 'month_name', 'quarter', 'year']]
date_df.reset_index(drop=True, inplace=True)
print(f"dim_date created. Rows: {len(date_df)}")


# --- 6. Populate dim_grade --- #
grades = pd.concat([main_df['sub1_r'], main_df['sub2_r'], main_df['sub3_r']]).unique()
grades = pd.DataFrame(grades, columns=['grade_code'])
grades.dropna(subset=['grade_code'], inplace=True)
grade_df = grades.copy()
grade_df['grade_description'] = grade_df['grade_code'] # Assuming description is same as code
grade_df['grade_points'] = None # No info in main_df
grade_df['pass_flag'] = None # No info in main_df
grade_df.reset_index(drop=True, inplace=True)
print(f"dim_grade created. Rows: {len(grade_df)}")


# --- 7. Prepare Fact Table Data --- #
# Unpivot subject/grade columns
fact_data = main_df.melt(
    id_vars=['index', 'stream', 'Zscore', 'district_rank', 'island_rank', 'al_year', 'day', 'month', 'year', 'gender'],
    value_vars=['sub1', 'sub2', 'sub3'],
    var_name='subject_col_name',
    value_name='subject_code'
)
fact_data['grade_col_name'] = fact_data['subject_col_name'].str.replace('sub', 'sub') + '_r'

# Merge grades back in
fact_data = fact_data.merge(
    main_df.melt(
        id_vars=['index'], # Use a common ID for merging
        value_vars=['sub1_r', 'sub2_r', 'sub3_r'],
        var_name='grade_col_name_merge',
        value_name='grade_code'
    ),
    left_on=['index', 'grade_col_name'],
    right_on=['index', 'grade_col_name_merge'],
    how='left'
)
fact_data.drop(columns=['subject_col_name', 'grade_col_name', 'grade_col_name_merge'], inplace=True)

# Clean up and ensure correct data types
fact_data.rename(columns={'Zscore': 'z_score'}, inplace=True)
fact_data['marks'] = fact_data['z_score'] # Assuming Zscore can be used as marks for now
fact_data['pass_flag'] = None # No direct info
fact_data['attempt_number'] = 1 # Assuming first attempt
fact_data['candidate_count'] = 1 # Assuming one candidate per row

# Link to Dimension Tables to get Keys

# Student Key
fact_data['student_id'] = fact_data.apply(
    lambda row: f"{row['gender']}_{pd.to_datetime(f"{row['year']}-{row['month']}-{row['day']}", errors='coerce').strftime('%Y%m%d')}"
    if pd.notna(pd.to_datetime(f"{row['year']}-{row['month']}-{row['day']}", errors='coerce')) else f"{row['gender']}_UNKNOWN_{row['day']}_{row['month']}_{row['year']}",
    axis=1
)
fact_data = fact_data.merge(student_df[['student_id']].reset_index().rename(columns={'index': 'student_key'}), on='student_id', how='left')

# Subject Key
fact_data = fact_data.merge(subject_df[['subject_code']].reset_index().rename(columns={'index': 'subject_key'}), on='subject_code', how='left')

# Stream Key
fact_data.rename(columns={'stream': 'stream_code'}, inplace=True)
fact_data = fact_data.merge(stream_df[['stream_code']].reset_index().rename(columns={'index': 'stream_key'}), on='stream_code', how='left')

# Location Key
fact_data.rename(columns={'district_rank': 'district'}, inplace=True)
fact_data = fact_data.merge(location_df[['district']].reset_index().rename(columns={'index': 'location_key'}), on='district', how='left')

# Date Key (Examination Date)
fact_data['full_date_exam'] = pd.to_datetime(fact_data['al_year'].astype(str) + '-01-01', errors='coerce')
fact_data['date_key'] = fact_data['full_date_exam'].dt.strftime('%Y%m%d').astype(int)
fact_data = fact_data.merge(date_df[['date_key']].reset_index().rename(columns={'index': 'date_key_dim'}), on='date_key', how='left')
fact_data.drop(columns=['date_key'], inplace=True) # Drop original date_key
fact_data.rename(columns={'date_key_dim': 'date_key'}, inplace=True) # Rename merged key


# Grade Key
fact_data = fact_data.merge(grade_df[['grade_code']].reset_index().rename(columns={'index': 'grade_key'}), on='grade_code', how='left')


fact_df = fact_data[[
    'student_key', 'subject_key', 'stream_key', 'location_key', 'date_key',
    'grade_key', 'marks', 'z_score', 'pass_flag', 'attempt_number',
    'candidate_count'
]].copy()
fact_df.dropna(subset=[
    'student_key', 'subject_key', 'stream_key', 'date_key'
], inplace=True) # Foreign keys that are NOT NULL in schema

fact_df.reset_index(drop=True, inplace=True)
print(f"fact_exam_result created. Rows: {len(fact_df)}")


/tmp/ipykernel_578/1803248824.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  student_df_raw['date_of_birth'] = pd.to_datetime(student_df_raw['date_of_birth_str'], errors='coerce')


dim_student created. Rows: 9681
dim_subject created. Rows: 62
dim_stream created. Rows: 8
dim_location created. Rows: 173078
dim_date created. Rows: 1
dim_grade created. Rows: 8


KeyboardInterrupt: 

In [ ]:
tables = {
    "dim_student": student_df,
    "dim_subject": subject_df,
    "dim_stream": stream_df,
    "dim_location": location_df,
    "dim_date": date_df,
    "dim_grade": grade_df,
    "fact_exam_result": fact_df
}

for table_name, dataframe in tables.items():
    dataframe.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False
    )
    print(f"Loaded: {table_name} | Rows: {len(dataframe)}")

print("All tables loaded successfully!")

In [18]:
tables = {
    "dim_student": student_df,
    "dim_subject": subject_df,
    "dim_stream": stream_df,
    "dim_location": location_df,
    "dim_date": date_df,
    "dim_grade": grade_df,
    "fact_exam_result": fact_df
}

# It's good practice to drop existing tables and re-create them when populating with new data
# or if the schema might have changed, especially during development.
# However, if 'if_exists="append"' is desired, ensure primary keys are handled correctly
# and that you don't duplicate data. For now, let's keep 'append' for SQLite, but
# be aware of potential duplicates if running multiple times without dropping.

for table_name, dataframe in tables.items():
    print(f"Loading data into {table_name}...")
    # SQLite automatically handles AUTOINCREMENT for INTEGER PRIMARY KEY
    # Resetting index to get a fresh integer index for the primary keys if needed
    # For dim tables, we don't pass `index=False` directly because the DB will assign keys
    if table_name == "dim_student":
      # For dimension tables, we usually let the database handle the PRIMARY KEY AUTOINCREMENT
      # so we exclude the 'index' when loading. We also drop the 'student_id' because it's only for temporary reference.
      dataframe.drop(columns=['student_id'], inplace=True, errors='ignore')

    if table_name == "dim_date":
      # dim_date has a specific date_key, so we set it as index to be used as PK
      dataframe.set_index('date_key', inplace=True)

    try:
        dataframe.to_sql(
            table_name,
            engine,
            if_exists="append",
            index=False if table_name != "dim_date" else True
        )
        print(f"Loaded: {table_name} | Rows: {len(dataframe)}")
    except Exception as e:
        print(f"Error loading {table_name}: {e}")

print("All tables load process completed!")

Loading data into dim_student...
Loaded: dim_student | Rows: 0
Loading data into dim_subject...
Loaded: dim_subject | Rows: 0
Loading data into dim_stream...
Loaded: dim_stream | Rows: 0
Loading data into dim_location...
Loaded: dim_location | Rows: 0
Loading data into dim_date...
Loaded: dim_date | Rows: 0
Loading data into dim_grade...
Loaded: dim_grade | Rows: 0
Loading data into fact_exam_result...
Loaded: fact_exam_result | Rows: 0
All tables load process completed!


In [20]:
!pip install -q pandas sqlalchemy matplotlib seaborn

In [21]:
import pandas as pd
import numpy as np
import io

from sqlalchemy import create_engine, text

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

print("Libraries imported successfully!")

Libraries imported successfully!


In [22]:
print("Uploaded files:")
for filename in uploaded.keys():
    print(filename)

Uploaded files:
2020_al_data_kaggle_upload_new_old_syllabi.csv


In [23]:
file_name = "2020_al_data_kaggle_upload_new_old_syllabi.csv"

file_content = uploaded[file_name]

main_df = pd.read_csv(
    io.BytesIO(file_content)
)

print("Dataset loaded successfully!")
print("Rows:", main_df.shape[0])
print("Columns:", main_df.shape[1])

display(main_df.head())


Dataset loaded successfully!
Rows: 337553
Columns: 19


,index,stream,Zscore,district_rank,island_rank,al_year,sub1,sub1_r,sub2,sub2_r,sub3,sub3_r,cgt_r,ge_r,syllabus,birth_day,birth_month,birth_year,gender
0,0,ARTS,-.3550,4336 (NEW),64994 (NEW),2020,POLITICAL SCIENCE,S,DANCING(BHARATHA),C,TAMIL,S,056,S,new,31,May,2001,female
1,1,ARTS,-.2648,4154 (NEW),62338 (NEW),2020,POLITICAL SCIENCE,S,CARNATIC MUSIC,C,TAMIL,C,032,C,new,13,January,2002,female
2,2,COMMERCE,-.4760,6910 (NEW),37307 (NEW),2020,ECONOMICS,S,BUSINESS STUDIES,S,ACCOUNTING,S,050,S,new,16,August,2001,female
3,3,COMMERCE,-.1012,5678 (NEW),30449 (NEW),2020,ECONOMICS,C,BUSINESS STUDIES,C,ACCOUNTING,S,034,S,new,16,August,2001,female
4,4,COMMERCE,.6014,3269 (NEW),17010 (NEW),2020,ECONOMICS,C,BUSINESS STUDIES,C,ACCOUNTING,B,036,S,new,7,August,2000,female


In [24]:
print(main_df.columns.tolist())

['index', 'stream', 'Zscore', 'district_rank', 'island_rank', 'al_year', 'sub1', 'sub1_r', 'sub2', 'sub2_r', 'sub3', 'sub3_r', 'cgt_r', 'ge_r', 'syllabus', 'birth_day', 'birth_month', 'birth_year', 'gender']


In [25]:
engine = create_engine(
    "sqlite:///al_exam_datawarehouse.db"
)

print("SQLite Data Warehouse created successfully!")

SQLite Data Warehouse created successfully!


In [38]:
main_df = main_df.copy()

# Remove accidental spaces from column names
main_df.columns = main_df.columns.str.strip()

# Convert important columns to numeric where possible
numeric_columns = [
    "Zscore",
    "district_rank",
    "island_rank",
    "al_year"
    # Removed 'birth_day', 'birth_month', 'birth_year' from here
    # as they need specific date parsing, not general numeric conversion.
]

for col in numeric_columns:
    # Check if the column exists before attempting conversion
    if col in main_df.columns:
        main_df[col] = pd.to_numeric(
            main_df[col],
            errors="coerce"
        )
    else:
        print(f"Warning: Column '{col}' not found in main_df. Skipping numeric conversion for this column.")


print("Data cleaning completed.")

Data cleaning completed.


In [27]:
drop_tables = """
DROP VIEW IF EXISTS olap_exam;

DROP TABLE IF EXISTS fact_exam_result;

DROP TABLE IF EXISTS dim_student;
DROP TABLE IF EXISTS dim_subject;
DROP TABLE IF EXISTS dim_stream;
DROP TABLE IF EXISTS dim_location;
DROP TABLE IF EXISTS dim_date;
DROP TABLE IF EXISTS dim_grade;
"""

with engine.begin() as conn:
    for statement in drop_tables.split(";"):
        if statement.strip():
            conn.execute(text(statement))

print("Old warehouse tables removed.")

Old warehouse tables removed.


In [29]:
create_dimensions = """

CREATE TABLE dim_student (
    student_key INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id VARCHAR(30) UNIQUE,
    gender VARCHAR(20),
    date_of_birth DATE
);

CREATE TABLE dim_subject (
    subject_key INTEGER PRIMARY KEY AUTOINCREMENT,
    subject_code VARCHAR(20),
    subject_name VARCHAR(100),
    subject_category VARCHAR(50)
);

CREATE TABLE dim_stream (
    stream_key INTEGER PRIMARY KEY AUTOINCREMENT,
    stream_code VARCHAR(20),
    stream_name VARCHAR(100),
    stream_category VARCHAR(100)
);

CREATE TABLE dim_location (
    location_key INTEGER PRIMARY KEY AUTOINCREMENT,
    district VARCHAR(100),
    province VARCHAR(100),
    examination_region VARCHAR(100)
);

CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date DATE,
    day INTEGER,
    month INTEGER,
    month_name VARCHAR(20),
    quarter INTEGER,
    year INTEGER
);

CREATE TABLE dim_grade (
    grade_key INTEGER PRIMARY KEY AUTOINCREMENT,
    grade_code VARCHAR(10),
    grade_description VARCHAR(50),
    grade_points DECIMAL(5,2),
    pass_flag BOOLEAN
);

"""

with engine.begin() as conn:

    for statement in create_dimensions.split(";"):

        if statement.strip():

            conn.execute(text(statement))

print("All dimension tables created successfully!")

All dimension tables created successfully!


In [61]:
student_df = main_df[
    ["index", "gender", "birth_day", "birth_month", "birth_year"]
].copy()

student_df = student_df.drop_duplicates(
    subset=["index"]
)

student_df["student_id"] = (
    student_df["index"]
    .astype(str)
)

# Convert numerical date components to nullable integers, coercing errors
student_df['birth_day_clean'] = pd.to_numeric(student_df['birth_day'], errors='coerce').astype(pd.Int64Dtype())
student_df['birth_year_clean'] = pd.to_numeric(student_df['birth_year'], errors='coerce').astype(pd.Int64Dtype())
# Keep birth_month as string, directly handle NaNs before strip
student_df['birth_month_clean'] = student_df['birth_month'].fillna('').astype(str).str.strip()

# Convert to string and handle nulls (pd.NA for Int64Dtype, empty string for str)
student_df['birth_day_str'] = student_df['birth_day_clean'].astype(str).replace('<NA>', '')
student_df['birth_year_str'] = student_df['birth_year_clean'].astype(str).replace('<NA>', '')
student_df['birth_month_str'] = student_df['birth_month_clean'] # NaNs already handled with fillna('')

# DEBUGGING: Print raw birth_month content before cleanup
print("Raw main_df['birth_month'] value counts:")
display(main_df['birth_month'].value_counts(dropna=False).head())

# If month or day string is empty, default to '01' (January 1st) to ensure parsability
# Also, ensure both are zero-padded to two digits for '%Y-%m-%d' format
student_df['birth_month_str'] = student_df['birth_month_str'].replace('', '01').str.zfill(2)
student_df['birth_day_str'] = student_df['birth_day_str'].replace('', '01').str.zfill(2) # Ensure day also has a default and is zero-padded

# Combine date components into a single string
# Using numeric month and day now
student_df['date_string'] = student_df['birth_year_str'] + '-' + \
                            student_df['birth_month_str'] + '-' + \
                            student_df['birth_day_str']

# Debugging: Print some date_string values and check for problematic components
print("Sample date_string values (first 5) after defaulting and zero-padding:")
display(student_df['date_string'].head())
print("Top 10 unique birth_month_str values and their counts after defaulting:")
display(student_df['birth_month_str'].value_counts().head(10))
print(f"Count of empty birth_year_str: {(student_df['birth_year_str'] == '').sum()}")
print(f"Count of empty birth_month_str: {(student_df['birth_month_str'] == '').sum()}")
print(f"Count of empty birth_day_str: {(student_df['birth_day_str'] == '').sum()}")

# Convert to datetime with explicit format
student_df["date_of_birth"] = pd.to_datetime(
    student_df['date_string'],
    format='%Y-%m-%d', # Now expecting numeric month and day
    errors="coerce"
)

# Debugging: Check how many NaT values are produced
print(f"Total NaT in date_of_birth after conversion: {student_df['date_of_birth'].isna().sum()}")

student_df = student_df[
    ["student_id", "gender", "date_of_birth"]
]

student_df = student_df.drop_duplicates(
    subset=["student_id"]
)

print("dim_student:")
print("Rows:", len(student_df))

display(student_df.head())

Raw main_df['birth_month'] value counts:


,count
birth_month,
NaN,337553


Sample date_string values (first 5) after defaulting and zero-padding:


,date_string
0,2001-01-31
1,2002-01-13
2,2001-01-16
3,2001-01-16
4,2000-01-07


Top 10 unique birth_month_str values and their counts after defaulting:


,count
birth_month_str,
01,337553


Count of empty birth_year_str: 1583
Count of empty birth_month_str: 0
Count of empty birth_day_str: 0
Total NaT in date_of_birth after conversion: 1589
dim_student:
Rows: 337553


,student_id,gender,date_of_birth
0,0,female,2001-01-31
1,1,female,2002-01-13
2,2,female,2001-01-16
3,3,female,2001-01-16
4,4,female,2000-01-07


In [32]:
student_df.to_sql(
    "dim_student",
    engine,
    if_exists="append",
    index=False
)

print("dim_student loaded successfully!")

dim_student loaded successfully!


In [33]:
subjects = pd.concat(
    [
        main_df["sub1"],
        main_df["sub2"],
        main_df["sub3"]
    ],
    ignore_index=True
)

subjects = (
    subjects
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
)

subject_df = pd.DataFrame({
    "subject_code": subjects,
    "subject_name": subjects,
    "subject_category": None
})

subject_df = subject_df.reset_index(drop=True)

print("dim_subject:")
print("Rows:", len(subject_df))

display(subject_df)

dim_subject:
Rows: 62


,subject_code,subject_name,subject_category
0,POLITICAL SCIENCE,POLITICAL SCIENCE,None
1,ECONOMICS,ECONOMICS,None
2,ISLAM,ISLAM,None
3,HISTORY OF SRI LANKA & EUROPE,HISTORY OF SRI LANKA & EUROPE,None
4,COMMUNICATION & MEDIA STUDIES,COMMUNICATION & MEDIA STUDIES,None
...,...,...,...
57,SANSKRIT,SANSKRIT,None
58,DRAMA AND THEATRE (ENGLISH),DRAMA AND THEATRE (ENGLISH),None
59,RUSSIAN,RUSSIAN,None
60,ARABIC,ARABIC,None


In [34]:
subject_df.to_sql(
    "dim_subject",
    engine,
    if_exists="append",
    index=False
)

print("dim_subject loaded successfully!")

dim_subject loaded successfully!


In [35]:
stream_df = (
    main_df[["stream"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

stream_df["stream"] = (
    stream_df["stream"]
    .astype(str)
    .str.strip()
)

stream_df = stream_df.rename(
    columns={
        "stream": "stream_code"
    }
)

stream_df["stream_name"] = stream_df["stream_code"]

stream_df["stream_category"] = None

stream_df = stream_df[
    [
        "stream_code",
        "stream_name",
        "stream_category"
    ]
]

print("dim_stream:")
print("Rows:", len(stream_df))

display(stream_df)

dim_stream:
Rows: 8


,stream_code,stream_name,stream_category
0,ARTS,ARTS,None
2,COMMERCE,COMMERCE,None
8,-,-,None
52,NON,NON,None
183,PHYSICAL SCIENCE,PHYSICAL SCIENCE,None
193,BIOLOGICAL SCIENCE,BIOLOGICAL SCIENCE,None
744,ENGINEERING TECHNOLOGY,ENGINEERING TECHNOLOGY,None
745,BIOSYSTEMS TECHNOLOGY,BIOSYSTEMS TECHNOLOGY,None


In [36]:
stream_df.to_sql(
    "dim_stream",
    engine,
    if_exists="append",
    index=False
)

print("dim_stream loaded successfully!")

dim_stream loaded successfully!


In [37]:
location_df = main_df[
    ["district_rank", "island_rank"]
].drop_duplicates().copy()

location_df = location_df.rename(
    columns={
        "district_rank": "district",
        "island_rank": "examination_region"
    }
)

location_df["province"] = None

location_df = location_df[
    [
        "district",
        "province",
        "examination_region"
    ]
]

print("dim_location:")
print("Rows:", len(location_df))

display(location_df.head(20))

dim_location:
Rows: 1


,district,province,examination_region
0,NaN,None,NaN


In [40]:
location_df.to_sql(
    "dim_location",
    engine,
    if_exists="append",
    index=False
)

print("dim_location loaded successfully!")

dim_location loaded successfully!


In [41]:
date_df = (
    main_df[["al_year"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

date_df["year"] = date_df["al_year"].astype(int)

date_df["full_date"] = pd.to_datetime(
    date_df["year"].astype(str) + "-01-01"
)

date_df["date_key"] = (
    date_df["full_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

date_df["day"] = date_df["full_date"].dt.day
date_df["month"] = date_df["full_date"].dt.month
date_df["month_name"] = date_df["full_date"].dt.month_name()
date_df["quarter"] = date_df["full_date"].dt.quarter

date_df = date_df[
    [
        "date_key",
        "full_date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year"
    ]
]

print("dim_date:")
display(date_df)

dim_date:


,date_key,full_date,day,month,month_name,quarter,year
0,20200101,2020-01-01,1,1,January,1,2020


In [42]:
date_df.to_sql(
    "dim_date",
    engine,
    if_exists="append",
    index=False
)

print("dim_date loaded successfully!")

dim_date loaded successfully!


In [43]:
grades = pd.concat(
    [
        main_df["sub1_r"],
        main_df["sub2_r"],
        main_df["sub3_r"]
    ],
    ignore_index=True
)

grades = (
    grades
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
)

grade_df = pd.DataFrame({
    "grade_code": grades,
    "grade_description": grades,
    "grade_points": None,
    "pass_flag": None
})

grade_df = grade_df.reset_index(drop=True)

print("dim_grade:")
print("Rows:", len(grade_df))

display(grade_df)

dim_grade:
Rows: 8


,grade_code,grade_description,grade_points,pass_flag
0,S,S,None,None
1,C,C,None,None
2,B,B,None,None
3,F,F,None,None
4,Absent,Absent,None,None
5,A,A,None,None
6,Withheld,Withheld,None,None
7,A.1,A.1,None,None


In [44]:
grade_df.to_sql(
    "dim_grade",
    engine,
    if_exists="append",
    index=False
)

print("dim_grade loaded successfully!")

dim_grade loaded successfully!


In [45]:
dimension_tables = [
    "dim_student",
    "dim_subject",
    "dim_stream",
    "dim_location",
    "dim_date",
    "dim_grade"
]

for table in dimension_tables:

    count = pd.read_sql(
        f"SELECT COUNT(*) AS count FROM {table}",
        engine
    ).iloc[0]["count"]

    print(f"{table}: {count:,} rows")

dim_student: 337,553 rows
dim_subject: 62 rows
dim_stream: 8 rows
dim_location: 1 rows
dim_date: 1 rows
dim_grade: 8 rows


In [47]:
create_fact = """

CREATE TABLE fact_exam_result (

    result_key INTEGER PRIMARY KEY AUTOINCREMENT,

    student_key INTEGER NOT NULL,
    subject_key INTEGER NOT NULL,
    stream_key INTEGER NOT NULL,
    location_key INTEGER,
    date_key INTEGER NOT NULL,
    grade_key INTEGER,

    marks DECIMAL(5,2),
    z_score DECIMAL(6,3),

    pass_flag BOOLEAN,
    attempt_number INTEGER,

    candidate_count INTEGER,

    FOREIGN KEY (student_key)
        REFERENCES dim_student(student_key),

    FOREIGN KEY (subject_key)
        REFERENCES dim_subject(subject_key),

    FOREIGN KEY (stream_key)
        REFERENCES dim_stream(stream_key),

    FOREIGN KEY (location_key)
        REFERENCES dim_location(location_key),

    FOREIGN KEY (date_key)
        REFERENCES dim_date(date_key),

    FOREIGN KEY (grade_key)
        REFERENCES dim_grade(grade_key)
);

"""

with engine.begin() as conn:
    conn.execute(text(create_fact))

print("Fact table created successfully!")

Fact table created successfully!


In [49]:
fact_data = main_df.melt(
    id_vars=[
        "index",
        "stream",
        "Zscore",
        "district_rank",
        "island_rank",
        "al_year",
        "gender"
    ],

    value_vars=[
        "sub1",
        "sub2",
        "sub3"
    ],

    var_name="subject_column",
    value_name="subject_code"
)

print("Subject records created:", len(fact_data))

display(fact_data.head(10))

Subject records created: 1012659


,index,stream,Zscore,district_rank,island_rank,al_year,gender,subject_column,subject_code
0,0,ARTS,-0.3550,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE
1,1,ARTS,-0.2648,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE
2,2,COMMERCE,-0.4760,NaN,NaN,2020,female,sub1,ECONOMICS
3,3,COMMERCE,-0.1012,NaN,NaN,2020,female,sub1,ECONOMICS
4,4,COMMERCE,0.6014,NaN,NaN,2020,female,sub1,ECONOMICS
5,5,ARTS,-0.5061,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE
6,6,ARTS,0.0177,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE
7,7,ARTS,-0.3106,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE
8,8,-,NaN,NaN,NaN,2020,female,sub1,ECONOMICS
9,9,COMMERCE,1.0447,NaN,NaN,2020,female,sub1,ECONOMICS


In [50]:
grade_mapping = main_df.melt(
    id_vars=["index"],

    value_vars=[
        "sub1_r",
        "sub2_r",
        "sub3_r"
    ],

    var_name="grade_column",
    value_name="grade_code"
)

grade_mapping["subject_column"] = (
    grade_mapping["grade_column"]
    .str.replace("_r", "", regex=False)
)

fact_data = fact_data.merge(
    grade_mapping[
        [
            "index",
            "subject_column",
            "grade_code"
        ]
    ],

    on=[
        "index",
        "subject_column"
    ],

    how="left"
)

display(fact_data.head(10))

,index,stream,Zscore,district_rank,island_rank,al_year,gender,subject_column,subject_code,grade_code
0,0,ARTS,-0.3550,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE,S
1,1,ARTS,-0.2648,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE,S
2,2,COMMERCE,-0.4760,NaN,NaN,2020,female,sub1,ECONOMICS,S
3,3,COMMERCE,-0.1012,NaN,NaN,2020,female,sub1,ECONOMICS,C
4,4,COMMERCE,0.6014,NaN,NaN,2020,female,sub1,ECONOMICS,C
5,5,ARTS,-0.5061,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE,S
6,6,ARTS,0.0177,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE,S
7,7,ARTS,-0.3106,NaN,NaN,2020,female,sub1,POLITICAL SCIENCE,S
8,8,-,NaN,NaN,NaN,2020,female,sub1,ECONOMICS,S
9,9,COMMERCE,1.0447,NaN,NaN,2020,female,sub1,ECONOMICS,B


In [51]:
fact_data["marks"] = None

fact_data["z_score"] = pd.to_numeric(
    fact_data["Zscore"],
    errors="coerce"
)

fact_data["pass_flag"] = (
    fact_data["grade_code"]
    .astype(str)
    .str.upper()
    .isin(["A", "B", "C", "S"])
)

fact_data["attempt_number"] = 1

fact_data["candidate_count"] = 1

In [52]:
student_keys = pd.read_sql(
    """
    SELECT
        student_key,
        student_id
    FROM dim_student
    """,
    engine
)

fact_data["student_id"] = (
    fact_data["index"]
    .astype(str)
)

fact_data = fact_data.merge(
    student_keys,
    on="student_id",
    how="left"
)

print("Student key mapping completed.")

Student key mapping completed.


In [53]:
subject_keys = pd.read_sql(
    """
    SELECT
        subject_key,
        subject_code
    FROM dim_subject
    """,
    engine
)

fact_data = fact_data.merge(
    subject_keys,
    on="subject_code",
    how="left"
)

print("Subject key mapping completed.")

Subject key mapping completed.


In [55]:
stream_keys = pd.read_sql(
    """
    SELECT
        stream_key,
        stream_code
    FROM dim_stream
    """,
    engine
)

fact_data["stream_code"] = (
    fact_data["stream"]
    .astype(str)
    .str.strip()
)

fact_data = fact_data.merge(
    stream_keys,
    on="stream_code",
    how="left"
)

print("Stream key mapping completed.")

Stream key mapping completed.


In [64]:
location_keys = pd.read_sql(
    """
    SELECT
        location_key,
        district,
        examination_region
    FROM dim_location
    """,
    engine
)

# Ensure district and examination_region in fact_data are strings for merging
fact_data["district"] = (
    fact_data["district_rank"]
    .astype(str)
    .replace('nan', None) # Replace string 'nan' with None for consistency
)

fact_data["examination_region"] = (
    fact_data["island_rank"]
    .astype(str)
    .replace('nan', None) # Replace string 'nan' with None
)

fact_data = fact_data.merge(
    location_keys,
    on=[
        "district",
        "examination_region"
    ],
    how="left"
)

print("Location key mapping completed.")

Location key mapping completed.


In [58]:
date_keys = pd.read_sql(
    """
    SELECT
        date_key,
        year
    FROM dim_date
    """,
    engine
)

fact_data["year"] = (
    fact_data["al_year"]
    .astype(int)
)

fact_data = fact_data.merge(
    date_keys,
    on="year",
    how="left"
)

print("Date key mapping completed.")

Date key mapping completed.


In [62]:
grade_keys = pd.read_sql(
    """
    SELECT
        grade_key,
        grade_code
    FROM dim_grade
    """,
    engine
)

fact_data = fact_data.merge(
    grade_keys,
    on="grade_code",
    how="left"
)

print("Grade key mapping completed.")

Grade key mapping completed.


In [70]:
# Ensure 'location_key' exists in fact_data before selection
# If it was not successfully merged in LMfwifryn0Hn, add it as nullable.
if 'location_key' not in fact_data.columns:
    fact_data['location_key'] = pd.NA

fact_df = fact_data[
    [
        "student_key",
        "subject_key",
        "stream_key",
        "location_key",
        "date_key",
        "grade_key",
        "marks",
        "z_score",
        "pass_flag",
        "attempt_number",
        "candidate_count"
    ]
].copy()

In [65]:
key_columns = [
    "student_key",
    "subject_key",
    "stream_key",
    "date_key"
]

for col in key_columns:

    missing = fact_df[col].isna().sum()

    print(
        f"{col}: {missing} missing"
    )

student_key: 0 missing
subject_key: 0 missing
stream_key: 0 missing
date_key: 0 missing


In [73]:
key_columns_to_check = [
    "student_key",
    "subject_key",
    "stream_key",
    "date_key",
    "location_key", # Also check location_key, as it can be nullable
    "grade_key" # Also check grade_key, as it can be nullable
]

print("Missing key counts in fact_df BEFORE dropna:")
for col in key_columns_to_check:
    missing = fact_df[col].isna().sum()
    print(f"{col}: {missing} missing")

fact_df = fact_df.dropna(
    subset=[
        "student_key",
        "subject_key",
        "stream_key",
        "date_key"
    ]
)

fact_df = fact_df.reset_index(drop=True)

print("Final fact rows:", len(fact_df))

Missing key counts in fact_df BEFORE dropna:
student_key: 0 missing
subject_key: 0 missing
stream_key: 0 missing
date_key: 0 missing
location_key: 0 missing
grade_key: 0 missing
Final fact rows: 1012659


In [77]:
# Print dtypes to check for any unexpected types before insertion
print("fact_df dtypes before to_sql:")
display(fact_df.dtypes)

try:
    # Use an explicit transaction to ensure proper commit/rollback and error reporting
    with engine.begin() as connection:
        fact_df.to_sql(
            "fact_exam_result",
            connection, # Use the connection from the transaction
            if_exists="append",
            index=False
        )
    print("fact_exam_result loaded successfully!")
except Exception as e:
    print(f"Error loading fact_exam_result: {e}")

# Verify the count immediately after the load attempt
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM fact_exam_result")).scalar()
    print(f"Direct query count for fact_exam_result after load attempt: {result:,} rows")

fact_df dtypes before to_sql:


,0
student_key,int64
subject_key,int64
stream_key,int64
location_key,int64
date_key,int64
grade_key,int64
marks,object
z_score,float64
pass_flag,bool
attempt_number,int64


fact_exam_result loaded successfully!
Direct query count for fact_exam_result after load attempt: 1,012,659 rows


In [68]:
all_tables = [
    "dim_student",
    "dim_subject",
    "dim_stream",
    "dim_location",
    "dim_date",
    "dim_grade",
    "fact_exam_result"
]

for table in all_tables:

    result = pd.read_sql(
        f"SELECT COUNT(*) AS count FROM {table}",
        engine
    )

    print(
        f"{table}: {result.iloc[0]['count']:,} rows"
    )

dim_student: 337,553 rows
dim_subject: 62 rows
dim_stream: 8 rows
dim_location: 1 rows
dim_date: 1 rows
dim_grade: 8 rows
fact_exam_result: 0 rows


In [71]:
create_olap_view = """

CREATE VIEW olap_exam AS

SELECT

    f.result_key,

    -- STUDENT
    s.student_id,
    s.gender,
    s.date_of_birth,

    -- SUBJECT
    sub.subject_code,
    sub.subject_name,
    sub.subject_category,

    -- STREAM
    st.stream_code,
    st.stream_name,
    st.stream_category,

    -- LOCATION
    l.district,
    l.province,
    l.examination_region,

    -- DATE
    d.date_key,
    d.full_date,
    d.day,
    d.month,
    d.month_name,
    d.quarter,
    d.year,

    -- GRADE
    g.grade_code,
    g.grade_description,
    g.grade_points,

    -- FACT MEASURES
    f.marks,
    f.z_score,
    f.pass_flag,
    f.attempt_number,
    f.candidate_count

FROM fact_exam_result f

JOIN dim_student s
    ON f.student_key = s.student_key

JOIN dim_subject sub
    ON f.subject_key = sub.subject_key

JOIN dim_stream st
    ON f.stream_key = st.stream_key

LEFT JOIN dim_location l
    ON f.location_key = l.location_key

JOIN dim_date d
    ON f.date_key = d.date_key

LEFT JOIN dim_grade g
    ON f.grade_key = g.grade_key

"""

with engine.begin() as conn:

    conn.execute(
        text(create_olap_view)
    )

print("OLAP view created successfully!")

OLAP view created successfully!


In [72]:
olap_df = pd.read_sql(
    "SELECT * FROM olap_exam",
    engine
)

print("OLAP records:", len(olap_df))

display(olap_df.head(10))

OLAP records: 0


,result_key,student_id,gender,date_of_birth,subject_code,subject_name,subject_category,stream_code,stream_name,stream_category,...,quarter,year,grade_code,grade_description,grade_points,marks,z_score,pass_flag,attempt_number,candidate_count


In [74]:
print(
    olap_df.columns.tolist()
)

['result_key', 'student_id', 'gender', 'date_of_birth', 'subject_code', 'subject_name', 'subject_category', 'stream_code', 'stream_name', 'stream_category', 'district', 'province', 'examination_region', 'date_key', 'full_date', 'day', 'month', 'month_name', 'quarter', 'year', 'grade_code', 'grade_description', 'grade_points', 'marks', 'z_score', 'pass_flag', 'attempt_number', 'candidate_count']


In [75]:
olap_cube = pd.pivot_table(
    olap_df,

    index="year",

    columns=[
        "stream_name",
        "subject_name"
    ],

    values="z_score",

    aggfunc="mean"
)

print("OLAP CUBE")
print("Year × Stream × Subject → Average Z-Score")

display(
    olap_cube.round(3)
)


OLAP CUBE
Year × Stream × Subject → Average Z-Score


year


In [76]:
rollup_year = pd.read_sql(
    """
    SELECT

        year,

        COUNT(*) AS candidate_count,

        ROUND(
            AVG(z_score),
            3
        ) AS average_z_score,

        SUM(
            CASE
                WHEN pass_flag = 1
                THEN 1
                ELSE 0
            END
        ) AS pass_count

    FROM olap_exam

    GROUP BY year

    ORDER BY year
    """,

    engine
)

rollup_year["pass_rate"] = (
    rollup_year["pass_count"]
    /
    rollup_year["candidate_count"]
    * 100
).round(2)

display(rollup_year)

,year,candidate_count,average_z_score,pass_count,pass_rate


In [78]:
drilldown = pd.read_sql(
    """
    SELECT

        year,
        quarter,
        month,
        month_name,

        COUNT(*) AS candidate_count,

        ROUND(
            AVG(z_score),
            3
        ) AS average_z_score

    FROM olap_exam

    GROUP BY
        year,
        quarter,
        month,
        month_name

    ORDER BY
        year,
        quarter,
        month
    """,

    engine
)

display(drilldown)

,year,quarter,month,month_name,candidate_count,average_z_score
0,2020,1,1,January,1012659,0.305


In [79]:
slice_2020 = pd.read_sql(
    """
    SELECT

        subject_name,

        COUNT(*) AS candidate_count,

        ROUND(
            AVG(z_score),
            3
        ) AS average_z_score

    FROM olap_exam

    WHERE year = 2020

    GROUP BY subject_name

    ORDER BY average_z_score DESC
    """,

    engine
)

display(slice_2020)

,subject_name,candidate_count,average_z_score
0,DRAMA AND THEATRE (ENGLISH),5,1.191
1,BUSINESS STATISTICS,1177,0.668
2,MATHEMATICS,748,0.528
3,ENGLISH,2031,0.520
4,FRENCH,1278,0.488
...,...,...,...
57,DRAMA AND THEATRE (TAMIL),2664,0.014
58,CHRISTIANITY,2893,-0.006
59,RUSSIAN,38,-0.066
60,MECHANICAL TECHNOLOGY,45,-0.068
